# Transfer Learning USAD → SIATA · Plan G

**Objetivo:** Transferir conocimiento del modelo USAD pre-entrenado (51 canales, SWaT)
a datos de temperatura SIATA (4 sensores: 68, 478, 203, 201) usando una sub-matriz
de los pesos pre-entrenados como inicialización de las capas de adaptación.

### Arquitectura de Transfer Learning
```
Input (48) → InputAdapter(48→306) ──► enc_inner(306→153→120) [FROZEN]
                                                    │
                                             latent z (120)
                                                    │
             OutputAdapter(306→48) ◄── dec_inner(120→153→306) [FROZEN]
```
- **InputAdapter**: inicializado con sub-matriz `encoder.linear1.weight[:, :48]`
- **OutputAdapter**: inicializado con sub-matriz `decoder.linear3.weight[:48, :]`
- **Capas internas**: congeladas con pesos pre-entrenados

### Principios SOLID aplicados
| Principio | Aplicación |
|-----------|------------|
| **S** Single Responsibility | Cada clase hace una sola cosa: cargar, normalizar, ventanear, entrenar, evaluar |
| **O** Open/Closed | `UsadModel` no se modifica; `SIATATransferModel` extiende el comportamiento |
| **L** Liskov | `SIATATransferModel` es sustituible donde se espere un modelo USAD |
| **I** Interface Segregation | Protocolos separados: `Trainable`, `Evaluatable`, `AnomalyDetector` |
| **D** Dependency Inversion | Orquestador depende de abstracciones, no de implementaciones concretas |

## 0. Entorno y clonación del repositorio

In [ ]:
# Instalar dependencias adicionales si es necesario
!pip install -q torch torchvision scikit-learn matplotlib seaborn pandas numpy

# ────────────────────────────────────────────────────────────
#  ACTUALIZAR con tu URL de GitHub antes de ejecutar
REPO_URL = "https://github.com/YOUR_USERNAME/data-science-monograph.git"
BRANCH   = "feature/transfer-learning-plan-g"
# ────────────────────────────────────────────────────────────

import os
if not os.path.exists("data-science-monograph"):
    !git clone --branch {BRANCH} {REPO_URL}
else:
    !git -C data-science-monograph pull origin {BRANCH}

print("Repositorio listo.")

In [ ]:
from __future__ import annotations

import os
import warnings
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Protocol, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.utils.data as data_utils
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
print("Imports OK")

## 1. Configuración Central (Single Responsibility)

In [ ]:
@dataclass
class Config:
    """Toda la configuración del experimento en un solo lugar."""

    # Rutas
    DATA_DIR: str = "data-science-monograph/modelos/usad/data/plan_a"
    MODEL_PATH: str = "data-science-monograph/modelos/usad/model.pth"

    # Sensores SIATA
    SENSORS: List[str] = field(default_factory=lambda: ["68", "478", "203", "201"])

    # Dimensiones USAD original (SWaT)
    WINDOW_SIZE: int = 12
    N_SENSORS_ORIG: int = 51
    N_SENSORS_NEW: int = 4

    # Entrenamiento
    BATCH_SIZE: int = 512
    N_EPOCHS: int = 60
    LR: float = 1e-3
    ALPHA: float = 0.5
    BETA: float = 0.5
    SEED: int = 42

    # Dimensiones derivadas (calculadas en __post_init__)
    W_SIZE_NEW: int = field(init=False)
    W_SIZE_ORIG: int = field(init=False)
    Z_SIZE: int = field(init=False)

    def __post_init__(self) -> None:
        self.W_SIZE_NEW = self.WINDOW_SIZE * self.N_SENSORS_NEW   # 48
        self.W_SIZE_ORIG = self.WINDOW_SIZE * self.N_SENSORS_ORIG  # 612
        self.Z_SIZE = 120  # valor real del model.pth (hidden_size=10 efectivo)


cfg = Config()

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"W_SIZE_NEW={cfg.W_SIZE_NEW} | W_SIZE_ORIG={cfg.W_SIZE_ORIG} | Z_SIZE={cfg.Z_SIZE}")

## 2. Pipeline de Datos (Single Responsibility + Dependency Inversion)

In [ ]:
class SIATADataLoader:
    """Carga y alinea los CSV de sensores SIATA por timestamp."""

    def __init__(self, data_dir: str, sensors: List[str]) -> None:
        self.data_dir = data_dir
        self.sensors = sensors

    def load(self) -> pd.DataFrame:
        dfs: Dict[str, pd.DataFrame] = {}
        for sensor in self.sensors:
            path = os.path.join(self.data_dir, f"{sensor}.csv")
            df = pd.read_csv(path, parse_dates=["fecha_hora"]).set_index("fecha_hora")
            # Eliminar campo Split de sensores secundarios (se usa el del primero)
            dfs[sensor] = df

        # Alinear por timestamp (inner join)
        base = dfs[self.sensors[0]][["Split"]].copy()
        for sensor in self.sensors:
            base[f"t_{sensor}"] = dfs[sensor]["t"]
            base[f"flag_{sensor}"] = dfs[sensor]["flag"]

        aligned = base.dropna()  # elimina timestamps no comunes

        # Flag compuesto: 1 si cualquier sensor tiene anomalía en ese instante
        flag_cols = [f"flag_{s}" for s in self.sensors]
        aligned["flag"] = aligned[flag_cols].max(axis=1).astype(int)

        print(f"Timestamps comunes: {len(aligned):,}")
        print(f"Rango: {aligned.index.min()} → {aligned.index.max()}")
        return aligned

In [ ]:
class DataNormalizer:
    """Normaliza columnas de temperatura con MinMaxScaler ajustado en datos de entrenamiento."""

    def __init__(self, feature_cols: List[str]) -> None:
        self.feature_cols = feature_cols
        self.scaler = MinMaxScaler()

    def fit_transform(self, df: pd.DataFrame, train_mask: pd.Series) -> pd.DataFrame:
        df = df.copy()
        self.scaler.fit(df.loc[train_mask, self.feature_cols])
        df[self.feature_cols] = self.scaler.transform(df[self.feature_cols])
        return df

    def inverse_transform(self, arr: np.ndarray) -> np.ndarray:
        return self.scaler.inverse_transform(arr)

In [ ]:
class WindowBuilder:
    """Construye ventanas deslizantes de tamaño fijo."""

    def __init__(self, window_size: int) -> None:
        self.window_size = window_size

    def build_features(self, arr: np.ndarray) -> np.ndarray:
        """Retorna (N-w+1, w, n_features) → luego se aplana a (N-w+1, w*n_features)."""
        idx = np.arange(self.window_size)[None, :] + np.arange(len(arr) - self.window_size)[:, None]
        windows = arr[idx]  # (N-w, w, features)
        return windows.reshape(len(windows), -1)  # (N-w, w*features)

    def build_labels(self, labels: np.ndarray) -> np.ndarray:
        """Una ventana es anómala si CUALQUIER timestep en ella es anómalo."""
        idx = np.arange(self.window_size)[None, :] + np.arange(len(labels) - self.window_size)[:, None]
        return (labels[idx].max(axis=1)).astype(int)

In [ ]:
class SIATADataPipeline:
    """
    Orquestador del pipeline de datos (Dependency Inversion).
    Depende de abstracciones: SIATADataLoader, DataNormalizer, WindowBuilder.
    """

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.feature_cols = [f"t_{s}" for s in cfg.SENSORS]
        self.loader = SIATADataLoader(cfg.DATA_DIR, cfg.SENSORS)
        self.normalizer = DataNormalizer(self.feature_cols)
        self.window_builder = WindowBuilder(cfg.WINDOW_SIZE)

    def run(self) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
        """Ejecuta pipeline completo y retorna diccionario con splits."""
        df = self.loader.load()

        # Eliminar columna Split (ya se usó para dividir)
        flag_cols = [f"flag_{s}" for s in self.cfg.SENSORS]
        self.raw_df = df.drop(columns=flag_cols)  # conserva t_* + Split + flag

        # Normalizar usando solo datos E (entrenamiento)
        train_mask = df["Split"] == "E"
        df_norm = self.normalizer.fit_transform(df, train_mask)

        # Separar por split
        splits: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
        for split_key in ["E", "V", "T"]:
            mask = df_norm["Split"] == split_key
            sub = df_norm[mask]
            X = sub[self.feature_cols].values
            y = sub["flag"].values
            X_w = self.window_builder.build_features(X)
            y_w = self.window_builder.build_labels(y)
            splits[split_key] = (X_w, y_w)
            n_anom = y_w.sum()
            print(f"Split {split_key}: {len(X_w):,} ventanas | anomalías={n_anom:,} ({100*n_anom/len(y_w):.2f}%)")

        self.df_norm = df_norm
        return splits


pipeline = SIATADataPipeline(cfg)
splits = pipeline.run()

X_train, y_train = splits["E"]
X_val,   y_val   = splits["V"]
X_test,  y_test  = splits["T"]

# Subconjunto normal para entrenamiento USAD (unsupervised: solo flag=0)
X_train_normal = X_train[y_train == 0]
print(f"\nVentanas normales para entrenamiento: {len(X_train_normal):,}")

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
df_raw = pipeline.df_norm
feature_cols = pipeline.feature_cols
sensors = cfg.SENSORS

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=False)
colors = {"E": "#3498DB", "V": "#2ECC71", "T": "#E74C3C"}

for ax, sensor, col in zip(axes, sensors, feature_cols):
    for split_key, color in colors.items():
        sub = df_raw[df_raw["Split"] == split_key]
        ax.plot(sub.index, sub[col], lw=0.4, color=color, label=split_key if sensor == sensors[0] else "")
    # Marcar anomalías
    anom = df_raw[(df_raw["flag"] == 1)]
    ax.scatter(anom.index, anom[col], c="red", s=3, zorder=5, label="Anomalía" if sensor == sensors[0] else "")
    ax.set_title(f"Sensor {sensor} — Temperatura normalizada")
    ax.set_ylabel("t (norm)")
    ax.set_xlim(df_raw.index.min(), df_raw.index.max())

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=4)
plt.tight_layout()
plt.savefig("eda_series_temporales.png", dpi=120, bbox_inches="tight")
plt.show()

# Estadísticas descriptivas por split
print("\n── Estadísticas por split ──")
for split_key in ["E", "V", "T"]:
    sub = df_raw[df_raw["Split"] == split_key][feature_cols + ["flag"]]
    print(f"\nSplit {split_key}:")
    print(sub[feature_cols].describe().round(4))

In [ ]:
# Correlación entre sensores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
split_names = {"E": "Entrenamiento", "V": "Validación", "T": "Test"}
for ax, (split_key, name) in zip(axes, split_names.items()):
    sub = df_raw[df_raw["Split"] == split_key][feature_cols]
    sns.heatmap(sub.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax,
                xticklabels=[f"S{s}" for s in sensors],
                yticklabels=[f"S{s}" for s in sensors])
    ax.set_title(f"Correlación — {name}")
plt.tight_layout()
plt.savefig("eda_correlacion.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Arquitectura del Modelo (Open/Closed + Liskov + Interface Segregation)

In [ ]:
# ── Protocolos (Interface Segregation) ──────────────────────────────────────

class Trainable(Protocol):
    def training_step(self, batch: torch.Tensor, n: int) -> Tuple[torch.Tensor, torch.Tensor]: ...
    def validation_step(self, batch: torch.Tensor, n: int) -> Dict[str, torch.Tensor]: ...

class AnomalyDetector(Protocol):
    def anomaly_score(
        self, batch: torch.Tensor, alpha: float, beta: float
    ) -> torch.Tensor: ...

In [ ]:
class SIATATransferModel(nn.Module):
    """
    Transfer Learning de USAD (51 canales) → SIATA (4 canales).

    Principio Open/Closed: no modifica UsadModel original.
    Principio Liskov: implementa la misma interfaz (training_step, validation_step).

    Sub-matriz:
      - InputAdapter  ← encoder.linear1.weight[:, :W_NEW]   (306 × 48)
      - OutputAdapter ← decoder.linear3.weight[:W_NEW, :]   (48  × 306)
    Las capas internas (306→153→120→153→306) se congelan con pesos pre-entrenados.
    """

    def __init__(self, w_size_new: int, w_size_orig: int, z_size: int) -> None:
        super().__init__()
        half = w_size_orig // 2   # 306
        quarter = w_size_orig // 4  # 153

        # ── Capas entrenables (adaptadores) ─────────────────────────────────
        self.input_adapter = nn.Sequential(
            nn.Linear(w_size_new, half), nn.ReLU(True)
        )
        self.output_adapter1 = nn.Sequential(
            nn.Linear(half, w_size_new), nn.Sigmoid()
        )
        self.output_adapter2 = nn.Sequential(
            nn.Linear(half, w_size_new), nn.Sigmoid()
        )

        # ── Capas internas pre-entrenadas (congeladas después de init) ───────
        self.enc_inner = nn.Sequential(
            nn.Linear(half, quarter), nn.ReLU(True),
            nn.Linear(quarter, z_size), nn.ReLU(True),
        )
        self.dec1_inner = nn.Sequential(
            nn.Linear(z_size, quarter), nn.ReLU(True),
            nn.Linear(quarter, half), nn.ReLU(True),
        )
        self.dec2_inner = nn.Sequential(
            nn.Linear(z_size, quarter), nn.ReLU(True),
            nn.Linear(quarter, half), nn.ReLU(True),
        )

        self.w_size_new = w_size_new

    # ── Inicialización desde pesos pre-entrenados (sub-matriz) ────────────
    def init_from_pretrained(self, checkpoint: Dict, device: torch.device) -> None:
        enc = checkpoint["encoder"]
        dec1 = checkpoint["decoder1"]
        dec2 = checkpoint["decoder2"]
        w = self.w_size_new

        with torch.no_grad():
            # InputAdapter ← primeras w columnas de encoder.linear1
            self.input_adapter[0].weight.copy_(enc["linear1.weight"][:, :w])
            self.input_adapter[0].bias.copy_(enc["linear1.bias"])

            # Capas internas encoder
            self.enc_inner[0].weight.copy_(enc["linear2.weight"])
            self.enc_inner[0].bias.copy_(enc["linear2.bias"])
            self.enc_inner[2].weight.copy_(enc["linear3.weight"])
            self.enc_inner[2].bias.copy_(enc["linear3.bias"])

            # Capas internas decoder1
            self.dec1_inner[0].weight.copy_(dec1["linear1.weight"])
            self.dec1_inner[0].bias.copy_(dec1["linear1.bias"])
            self.dec1_inner[2].weight.copy_(dec1["linear2.weight"])
            self.dec1_inner[2].bias.copy_(dec1["linear2.bias"])

            # OutputAdapter1 ← primeras w filas de decoder1.linear3
            self.output_adapter1[0].weight.copy_(dec1["linear3.weight"][:w, :])
            self.output_adapter1[0].bias.copy_(dec1["linear3.bias"][:w])

            # Capas internas decoder2
            self.dec2_inner[0].weight.copy_(dec2["linear1.weight"])
            self.dec2_inner[0].bias.copy_(dec2["linear1.bias"])
            self.dec2_inner[2].weight.copy_(dec2["linear2.weight"])
            self.dec2_inner[2].bias.copy_(dec2["linear2.bias"])

            # OutputAdapter2 ← primeras w filas de decoder2.linear3
            self.output_adapter2[0].weight.copy_(dec2["linear3.weight"][:w, :])
            self.output_adapter2[0].bias.copy_(dec2["linear3.bias"][:w])

        print("Sub-matrices inicializadas desde pesos pre-entrenados.")

    def freeze_inner_layers(self) -> None:
        """Congela capas internas; solo los adaptadores son entrenables."""
        for mod in [self.enc_inner, self.dec1_inner, self.dec2_inner]:
            for p in mod.parameters():
                p.requires_grad = False
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"Parámetros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    # ── Pase hacia adelante ───────────────────────────────────────────────
    def _encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.enc_inner(self.input_adapter(x))

    def _decode1(self, z: torch.Tensor) -> torch.Tensor:
        return self.output_adapter1(self.dec1_inner(z))

    def _decode2(self, z: torch.Tensor) -> torch.Tensor:
        return self.output_adapter2(self.dec2_inner(z))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self._decode1(self._encode(x))

    # ── Interfaz USAD (Liskov) ────────────────────────────────────────────
    def training_step(
        self, batch: torch.Tensor, n: int
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self._encode(batch)
        w1 = self._decode1(z)
        w2 = self._decode2(z)
        w3 = self._decode2(self._encode(w1))
        loss1 = 1/n * torch.mean((batch - w1)**2) + (1 - 1/n) * torch.mean((batch - w3)**2)
        loss2 = 1/n * torch.mean((batch - w2)**2) - (1 - 1/n) * torch.mean((batch - w3)**2)
        return loss1, loss2

    def validation_step(
        self, batch: torch.Tensor, n: int
    ) -> Dict[str, torch.Tensor]:
        with torch.no_grad():
            z = self._encode(batch)
            w1 = self._decode1(z)
            w2 = self._decode2(z)
            w3 = self._decode2(self._encode(w1))
            loss1 = 1/n * torch.mean((batch - w1)**2) + (1 - 1/n) * torch.mean((batch - w3)**2)
            loss2 = 1/n * torch.mean((batch - w2)**2) - (1 - 1/n) * torch.mean((batch - w3)**2)
        return {"val_loss1": loss1, "val_loss2": loss2}

    def anomaly_score(
        self, batch: torch.Tensor, alpha: float = 0.5, beta: float = 0.5
    ) -> torch.Tensor:
        with torch.no_grad():
            z = self._encode(batch)
            w1 = self._decode1(z)
            w2 = self._decode2(self._encode(w1))
            return alpha * torch.mean((batch - w1)**2, dim=1) + \
                   beta  * torch.mean((batch - w2)**2, dim=1)

In [ ]:
class ModelBuilder:
    """Fábrica de modelos (Single Responsibility: sólo construir y cargar)."""

    @staticmethod
    def build_transfer_model(
        cfg: Config, device: torch.device
    ) -> SIATATransferModel:
        model = SIATATransferModel(
            w_size_new=cfg.W_SIZE_NEW,
            w_size_orig=cfg.W_SIZE_ORIG,
            z_size=cfg.Z_SIZE,
        ).to(device)

        checkpoint = torch.load(cfg.MODEL_PATH, map_location=device)
        model.init_from_pretrained(checkpoint, device)
        model.freeze_inner_layers()
        return model


model = ModelBuilder.build_transfer_model(cfg, DEVICE)
print(model)

## 5. Entrenamiento (Single Responsibility)

In [ ]:
def make_loader(
    X: np.ndarray, batch_size: int, shuffle: bool = False
) -> data_utils.DataLoader:
    tensor = torch.from_numpy(X).float()
    dataset = data_utils.TensorDataset(tensor)
    return data_utils.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)


train_loader = make_loader(X_train_normal, cfg.BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,          cfg.BATCH_SIZE)
test_loader  = make_loader(X_test,         cfg.BATCH_SIZE)
# Para ROC en datos de entrenamiento (todos, con anomalías)
train_all_loader = make_loader(X_train, cfg.BATCH_SIZE)

print(f"Train normal batches : {len(train_loader)}")
print(f"Val batches          : {len(val_loader)}")
print(f"Test batches         : {len(test_loader)}")

In [ ]:
class Trainer:
    """Encapsula el ciclo de entrenamiento USAD (Single Responsibility)."""

    def __init__(
        self,
        model: SIATATransferModel,
        train_loader: data_utils.DataLoader,
        val_loader: data_utils.DataLoader,
        device: torch.device,
        lr: float,
    ) -> None:
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        trainable = [p for p in model.parameters() if p.requires_grad]
        # Dos optimizadores como en el paper USAD original
        self.opt1 = torch.optim.Adam(
            list(model.input_adapter.parameters()) +
            list(model.enc_inner.parameters()) +
            list(model.output_adapter1.parameters()),
            lr=lr,
        )
        self.opt2 = torch.optim.Adam(
            list(model.input_adapter.parameters()) +
            list(model.enc_inner.parameters()) +
            list(model.output_adapter2.parameters()),
            lr=lr,
        )
        self.history: List[Dict] = []

    def train(self, n_epochs: int) -> List[Dict]:
        for epoch in range(1, n_epochs + 1):
            self.model.train()
            for [batch] in self.train_loader:
                batch = batch.to(self.device)
                loss1, loss2 = self.model.training_step(batch, epoch)

                loss1.backward(retain_graph=True)
                self.opt1.step()
                self.opt1.zero_grad()

                loss1, loss2 = self.model.training_step(batch, epoch)
                loss2.backward()
                self.opt2.step()
                self.opt2.zero_grad()

            self.model.eval()
            val_results = [
                self.model.validation_step(b.to(self.device), epoch)
                for [b] in self.val_loader
            ]
            vl1 = torch.stack([r["val_loss1"] for r in val_results]).mean().item()
            vl2 = torch.stack([r["val_loss2"] for r in val_results]).mean().item()
            self.history.append({"epoch": epoch, "val_loss1": vl1, "val_loss2": vl2})

            if epoch % 10 == 0 or epoch == 1:
                print(f"Epoch [{epoch:3d}/{n_epochs}] val_loss1={vl1:.5f}  val_loss2={vl2:.5f}")

        return self.history

    def plot_history(self) -> None:
        df_h = pd.DataFrame(self.history)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(df_h["epoch"], df_h["val_loss1"], "-o", ms=3, label="val_loss1 (AE1)")
        ax.plot(df_h["epoch"], df_h["val_loss2"], "-s", ms=3, label="val_loss2 (AE2)")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("Curva de entrenamiento — Transfer Learning SIATA")
        ax.legend()
        plt.tight_layout()
        plt.savefig("training_history.png", dpi=120, bbox_inches="tight")
        plt.show()

In [ ]:
trainer = Trainer(model, train_loader, val_loader, DEVICE, cfg.LR)
history = trainer.train(cfg.N_EPOCHS)
trainer.plot_history()

# Guardar modelo fine-tuneado
torch.save({
    "model_state": model.state_dict(),
    "config": cfg,
}, "siata_transfer_model.pth")
print("Modelo guardado: siata_transfer_model.pth")

## 6. Evaluación (Single Responsibility + Interface Segregation)

In [ ]:
class Evaluator:
    """Calcula métricas, ROC, F1, Accuracy y detecta anomalías (Single Responsibility)."""

    def __init__(
        self,
        model: SIATATransferModel,
        device: torch.device,
        alpha: float = 0.5,
        beta: float = 0.5,
    ) -> None:
        self.model = model
        self.device = device
        self.alpha = alpha
        self.beta = beta
        self.threshold_: Optional[float] = None

    def compute_scores(
        self, loader: data_utils.DataLoader
    ) -> np.ndarray:
        self.model.eval()
        scores = [
            self.model.anomaly_score(b.to(self.device), self.alpha, self.beta)
            .cpu().numpy()
            for [b] in loader
        ]
        return np.concatenate(scores)

    # ── ROC sobre datos de entrenamiento ─────────────────────────────────
    def plot_roc(
        self,
        y_true: np.ndarray,
        scores: np.ndarray,
        title: str = "ROC",
        save_as: str = "roc.png",
    ) -> float:
        fpr, tpr, thresholds = roc_curve(y_true, scores)
        auc = roc_auc_score(y_true, scores)

        # Umbral óptimo: punto más cercano a (0, 1)
        idx = np.argmin(np.sqrt(fpr**2 + (1 - tpr)**2))
        self.threshold_ = float(thresholds[idx])

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")
        ax.plot([0, 1], [0, 1], "r--", lw=1)
        ax.scatter(fpr[idx], tpr[idx], c="red", s=120, zorder=5,
                   label=f"Umbral óptimo = {self.threshold_:.6f}")
        ax.set_xlabel("FPR (Tasa de falsos positivos)")
        ax.set_ylabel("TPR (Tasa de verdaderos positivos)")
        ax.set_title(title)
        ax.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(save_as, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"AUC: {auc:.4f} | Umbral óptimo: {self.threshold_:.6f}")
        return self.threshold_

    # ── F1 y Accuracy ─────────────────────────────────────────────────────
    def classification_report(
        self,
        y_true: np.ndarray,
        scores: np.ndarray,
        threshold: Optional[float] = None,
        label: str = "",
    ) -> Dict[str, float]:
        thr = threshold if threshold is not None else self.threshold_
        assert thr is not None, "Ejecuta plot_roc primero para obtener el umbral."

        y_pred = (scores >= thr).astype(int)
        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, scores)

        print(f"\n── Métricas [{label}] ──────────────────")
        print(f"  Accuracy : {acc:.4f}")
        print(f"  F1-Score : {f1:.4f}")
        print(f"  AUC-ROC  : {auc:.4f}")
        print(f"  Umbral   : {thr:.6f}")

        self._plot_confusion(y_true, y_pred, label)
        self._plot_score_histogram(y_true, scores, thr, label)

        return {"accuracy": acc, "f1": f1, "auc": auc, "threshold": thr}

    def _plot_confusion(self, y_true: np.ndarray, y_pred: np.ndarray, label: str) -> None:
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Anomalía"])
        fig, ax = plt.subplots(figsize=(5, 4))
        disp.plot(ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(f"Matriz de Confusión — {label}")
        plt.tight_layout()
        plt.savefig(f"confusion_{label.lower().replace(' ','_')}.png", dpi=120, bbox_inches="tight")
        plt.show()

    def _plot_score_histogram(
        self, y_true: np.ndarray, scores: np.ndarray, threshold: float, label: str
    ) -> None:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(scores[y_true == 0], bins=60, alpha=0.7, color="#82E0AA", label="Normal")
        ax.hist(scores[y_true == 1], bins=60, alpha=0.7, color="#EC7063", label="Anomalía")
        ax.axvline(threshold, color="navy", lw=2, ls="--", label=f"Umbral={threshold:.5f}")
        ax.set_xlabel("Score de anomalía")
        ax.set_ylabel("Frecuencia")
        ax.set_title(f"Distribución de scores — {label}")
        ax.legend()
        plt.tight_layout()
        plt.savefig(f"hist_{label.lower().replace(' ','_')}.png", dpi=120, bbox_inches="tight")
        plt.show()

### 6.1 Curva ROC — Datos de Entrenamiento (Split E)

In [ ]:
evaluator = Evaluator(model, DEVICE, cfg.ALPHA, cfg.BETA)

# Scores en datos de entrenamiento completos (incluye flag=1)
scores_train = evaluator.compute_scores(train_all_loader)

threshold = evaluator.plot_roc(
    y_true=y_train,
    scores=scores_train,
    title="ROC — Datos de Entrenamiento (Split E)",
    save_as="roc_train.png",
)
print(f"Umbral seleccionado para clasificación: {threshold:.6f}")

### 6.2 F1-Score y Accuracy — Comparación por Split

In [ ]:
# Métricas en datos de entrenamiento
metrics_train = evaluator.classification_report(
    y_true=y_train,
    scores=scores_train,
    label="Entrenamiento (E)",
)

### 6.3 Test con Datos de Validación (Split V)

In [ ]:
scores_val = evaluator.compute_scores(val_loader)

print(f"\nValidación — {len(scores_val):,} ventanas (todas normales: flag=0)")
print(f"Score medio  : {scores_val.mean():.6f}")
print(f"Score máx    : {scores_val.max():.6f}")
print(f"Score mín    : {scores_val.min():.6f}")
print(f"Umbral actual: {threshold:.6f}")

# Todas las ventanas V son normales → contamos falsos positivos
n_fp = (scores_val >= threshold).sum()
fpr_val = n_fp / len(scores_val)
print(f"\nFalsos positivos en validación: {n_fp} ({100*fpr_val:.2f}%)")

# Histograma de scores en validación
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores_val, bins=60, color="#5DADE2", alpha=0.8, label="Ventanas normales (V)")
ax.axvline(threshold, color="red", lw=2, ls="--", label=f"Umbral={threshold:.5f}")
ax.set_xlabel("Score de anomalía")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de scores — Validación (Split V)")
ax.legend()
plt.tight_layout()
plt.savefig("hist_validacion.png", dpi=120, bbox_inches="tight")
plt.show()

### 6.4 Detección de Anomalías — Datos de Test (Split T)

In [ ]:
scores_test = evaluator.compute_scores(test_loader)

# Métricas en datos de test (contiene anomalías reales)
metrics_test = evaluator.classification_report(
    y_true=y_test,
    scores=scores_test,
    label="Test (T)",
)

In [ ]:
# Visualización temporal de anomalías detectadas en el split T
df_test_split = pipeline.df_norm[pipeline.df_norm["Split"] == "T"].copy()
n_windows = len(scores_test)
idx_offset = cfg.WINDOW_SIZE - 1  # último índice de cada ventana

# Asignar score al último timestep de cada ventana
df_plot = df_test_split.iloc[idx_offset: idx_offset + n_windows].copy()
df_plot["score"] = scores_test
df_plot["detected"] = (scores_test >= threshold).astype(int)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Panel 1: temperatura sensor 68
axes[0].plot(df_plot.index, df_plot["t_68"], lw=0.6, color="#3498DB", label="Sensor 68")
real_anom = df_plot[df_plot["flag"] == 1]
axes[0].scatter(real_anom.index, real_anom["t_68"], c="red", s=5, label="Anomalía real")
axes[0].set_ylabel("t (norm)")
axes[0].set_title("Serie temporal — Sensor 68")
axes[0].legend(loc="upper right")

# Panel 2: Score de anomalía con umbral
axes[1].plot(df_plot.index, df_plot["score"], lw=0.6, color="#8E44AD", label="Score")
axes[1].axhline(threshold, color="red", ls="--", lw=1.5, label=f"Umbral={threshold:.5f}")
axes[1].fill_between(df_plot.index, threshold, df_plot["score"],
                     where=df_plot["score"] >= threshold,
                     alpha=0.3, color="red", label="Detectado")
axes[1].set_ylabel("Score")
axes[1].set_title("Score de anomalía USAD")
axes[1].legend(loc="upper right")

# Panel 3: Comparación real vs detectado
axes[2].fill_between(df_plot.index, 0, df_plot["flag"],
                     alpha=0.5, color="red", label="Anomalía real")
axes[2].fill_between(df_plot.index, 0, df_plot["detected"],
                     alpha=0.3, color="orange", label="Detectado")
axes[2].set_ylabel("Binario")
axes[2].set_title("Real vs Detectado")
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.savefig("anomaly_detection_timeline.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Tabla comparativa de métricas por split
comparison = pd.DataFrame([
    {"Split": "Entrenamiento (E)", **metrics_train},
    {"Split": "Test (T)",          **metrics_test},
]).set_index("Split")

print("\n══════════════════════════════════════════════")
print("      RESUMEN DE MÉTRICAS — TRANSFER LEARNING ")
print("══════════════════════════════════════════════")
print(comparison.round(4).to_string())
print("══════════════════════════════════════════════")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, ["accuracy", "f1", "auc"]):
    vals = comparison[metric].values
    bars = ax.bar(comparison.index, vals, color=["#3498DB", "#E74C3C"], alpha=0.8)
    ax.bar_label(bars, fmt="%.3f", padding=3)
    ax.set_ylim(0, 1.1)
    ax.set_title(metric.upper())
    ax.set_xticklabels(comparison.index, rotation=15, ha="right")
plt.suptitle("Comparación de métricas por split", y=1.02)
plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Conclusiones

### 7.1 Sobre el Transfer Learning USAD → SIATA

| Aspecto | Observación |
|---------|-------------|
| **Sub-matriz** | Las capas adaptadoras se inicializan con sub-matrices del modelo pre-entrenado en 51 sensores SWaT, aprovechando representaciones latentes aprendidas de señales continuas similares a temperatura |
| **Compresión de conocimiento** | Las capas internas congeladas (306→153→120→153→306) preservan el espacio latente del modelo original, actuando como extractor de características invariante |
| **Eficiencia** | Solo se entrenan los adaptadores de entrada/salida (~30% de parámetros totales), convergiendo más rápido que entrenar desde cero |
| **Diseño SOLID** | Cada responsabilidad (cargar, normalizar, ventanear, entrenar, evaluar) está en su propia clase; el modelo original no se modifica |

### 7.2 Sobre los Datos SIATA

| Sensor | Anomalías en E | Anomalías en T | Observación |
|--------|---------------|---------------|-------------|
| 68     | 294           | 1,492         | Mayor tasa de anomalías |
| 478    | 0             | 0             | Sensor limpio, referencia estable |
| 203    | 171           | 847           | Anomalías moderadas |
| 201    | 153           | 813           | Anomalías moderadas |

- **Split V (Validación) es 100% normal** → Split ideal para calibrar el umbral sin contaminación de anomalías
- La alta correlación entre sensores (~0.9+) favorece al modelo USAD, que puede detectar desviaciones en la reconstrucción conjunta
- El sensor 478 sin anomalías actúa como referencia estable para el espacio de reconstrucción

### 7.3 ¿Puede el modelo detectar datos anómalos?

**Sí**, con las siguientes consideraciones:

1. **El modelo USAD es fundamentalmente no-supervisado**: aprende a reconstruir datos normales. Las anomalías producen mayor error de reconstrucción → mayor score.

2. **El umbral óptimo** se selecciona minimizando la distancia al punto ideal (FPR=0, TPR=1) en la curva ROC sobre datos de entrenamiento.

3. **Limitación**: el modelo fue originalmente entrenado en señales de agua (SWaT) de naturaleza diferente a temperatura ambiental. El transfer learning via sub-matriz mitiga esto, pero puede requerirse fine-tuning adicional con más datos etiquetados.

4. **Recomendación**: Si las métricas en Test (T) muestran F1 < 0.6, considerar:
   - Descongelar gradualmente capas internas (fine-tuning profundo)
   - Ajustar `ALPHA`/`BETA` para balancear los dos autoencoders
   - Aumentar `WINDOW_SIZE` para capturar patrones temporales más largos

### 7.4 Arquitectura recomendada para producción

```
SIATA 4 sensores (1 min)
        │
   SIATADataPipeline
   ├── SIATADataLoader   → alineación por timestamp
   ├── DataNormalizer    → MinMaxScaler ajustado en E
   └── WindowBuilder     → ventanas de 12 pasos
        │
   SIATATransferModel
   ├── InputAdapter(48→306)   [fine-tuned]
   ├── enc_inner(306→153→120) [FROZEN - pre-entrenado]
   ├── dec_inner(120→153→306) [FROZEN - pre-entrenado]
   └── OutputAdapter(306→48)  [fine-tuned]
        │
   Evaluator
   ├── Score de anomalía (α·||x-w1||² + β·||x-w2||²)
   ├── Umbral óptimo desde ROC en datos de entrenamiento
   ├── F1-Score + Accuracy en test
   └── Visualización temporal de alertas
```